# Task 1: Data Preprocessing for Network Analysis

Prepare GSE50081 NSCLC expression data for co-expression network construction using Louvain community detection.

**Dataset**: GSE50081 (Non-Small Cell Lung Cancer, 181 tumor samples)

## 1. Setup & Configuration

In [13]:
"""Setup and imports for Lab 7 Task 1 - Data Preprocessing."""
import logging
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")

Python 3.12.3 (main, Nov  6 2025, 13:44:16) [GCC 13.3.0]
numpy 2.1.3
pandas 2.2.3


In [14]:
@dataclass
class Config:
    """Configuration for preprocessing pipeline."""

    lab_data: Path = Path("../../../data/work/AndreiCod/lab06")
    expression_csv: Path = None
    export_dir: Path = Path("./artifacts")
    top_variable_genes: int = 5000
    variance_threshold: float = 0.1
    apply_log2: bool = False
    target_gene: str = "TP53"

    def __post_init__(self):
        self.expression_csv = self.lab_data / "GSE50081_gene_expression.csv"
        self.export_dir.mkdir(parents=True, exist_ok=True)


CONFIG = Config()
print(f"Data source: {CONFIG.expression_csv}")
print(f"Export dir: {CONFIG.export_dir.resolve()}")

Data source: ../../../data/work/AndreiCod/lab06/GSE50081_gene_expression.csv
Export dir: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts


## 2. Load Expression Data

In [15]:
if not CONFIG.expression_csv.exists():
    print(f"ERROR: Expression data not found at {CONFIG.expression_csv}")
    print("Please run Lab 6 first to download GSE50081 data.")
else:
    print(f"[OK] Found expression data: {CONFIG.expression_csv}")

[OK] Found expression data: ../../../data/work/AndreiCod/lab06/GSE50081_gene_expression.csv


In [16]:
def load_expression_matrix(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    logging.info(
        "Loaded expression matrix: %d genes x %d samples", df.shape[0], df.shape[1]
    )
    return df


expr_raw = load_expression_matrix(CONFIG.expression_csv)
expr_raw.head()

18:40:59 | INFO | Loaded expression matrix: 2001 genes x 181 samples


,GSM1213669,GSM1213670,GSM1213671,GSM1213672,GSM1213673,GSM1213674,GSM1213675,GSM1213676,GSM1213677,GSM1213678,...,GSM1213840,GSM1213841,GSM1213842,GSM1213843,GSM1213844,GSM1213845,GSM1213846,GSM1213847,GSM1213848,GSM1213849
Gene,,,,,,,,,,,,,,,,,,,,,
KRT6A,12.748584,7.600682,2.522234,3.447513,5.289046,12.484240,12.487501,3.166682,12.310659,7.223532,...,2.805136,13.205127,10.422477,12.365052,7.767332,5.447799,9.245193,2.882689,12.446786,10.500599
SPRR1B,9.031516,9.583801,3.511168,8.656000,3.540976,11.032316,11.894948,3.646791,10.078840,3.445245,...,3.202791,13.705600,8.037607,9.329184,10.089060,9.008952,10.952269,4.466389,9.148314,8.220998
SCGB1A1,11.270704,6.439259,11.908472,3.578830,3.733639,6.615157,6.551806,4.529208,3.324227,9.495581,...,5.133556,4.100377,9.786214,4.064680,4.383852,5.836540,11.427470,8.748188,8.060045,9.608806
RPS4Y1,11.414617,4.690488,11.788965,4.540654,10.699047,11.721166,5.859753,8.783656,4.480617,4.657880,...,11.376956,9.229483,4.401298,12.263688,10.490783,6.078063,4.740944,4.497662,4.368025,4.180693
SPINK1,5.698196,11.904706,9.032240,7.741032,7.089012,4.170886,3.323415,3.509905,3.562707,12.295183,...,4.680854,7.370971,11.072242,5.451755,9.895093,7.162684,7.385929,6.839303,6.015390,11.157566


In [17]:
if CONFIG.target_gene in expr_raw.index:
    print(f"*** Target gene {CONFIG.target_gene} FOUND! ***")
    tp53_expr = expr_raw.loc[CONFIG.target_gene]
    print(f"  Expression range: {tp53_expr.min():.2f} - {tp53_expr.max():.2f}")
else:
    print(f"WARNING: {CONFIG.target_gene} not found!")

*** Target gene TP53 FOUND! ***
  Expression range: 3.81 - 7.78


## 3. Preprocessing Functions

In [18]:
def apply_log2_transform(df: pd.DataFrame) -> pd.DataFrame:
    transformed = np.log2(df + 1)
    logging.info("Applied log2(x+1) transformation")
    return transformed


def select_top_variable_genes(
    df: pd.DataFrame, n_top: int, ensure_gene: str = None
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    variances = df.var(axis=1)
    variance_df = pd.DataFrame({"Gene": df.index, "Variance": variances}).sort_values(
        "Variance", ascending=False
    )
    top_genes = variance_df.head(n_top)["Gene"].tolist()
    if ensure_gene and ensure_gene in df.index and ensure_gene not in top_genes:
        top_genes.append(ensure_gene)
        logging.info(f"Added {ensure_gene} to gene set")
    df_top = df.loc[top_genes]
    logging.info(f"Selected top {n_top} variable genes")
    return df_top, variance_df


def filter_low_variance(
    df: pd.DataFrame, threshold: float, ensure_gene: str = None
) -> pd.DataFrame:
    variances = df.var(axis=1)
    mask = variances >= threshold
    if ensure_gene and ensure_gene in df.index:
        mask[ensure_gene] = True
    filtered = df.loc[mask]
    logging.info("Filtered genes: %d -> %d", len(df), len(filtered))
    return filtered

In [19]:
print("=" * 60)
print("PREPROCESSING PIPELINE - GSE50081 NSCLC Data")
print("=" * 60)
print(f"Shape: {expr_raw.shape[0]} genes x {expr_raw.shape[1]} samples")
print(f"Value range: [{expr_raw.values.min():.2f}, {expr_raw.values.max():.2f}]")

PREPROCESSING PIPELINE - GSE50081 NSCLC Data
Shape: 2001 genes x 181 samples
Value range: [1.92, 14.53]


In [20]:
if CONFIG.apply_log2:
    expr_log = apply_log2_transform(expr_raw)
else:
    expr_log = expr_raw.copy()
    print("Data already log2-normalized (Affymetrix RMA)")

Data already log2-normalized (Affymetrix RMA)


In [21]:
expr_top, variance_stats = select_top_variable_genes(
    expr_log, CONFIG.top_variable_genes, ensure_gene=CONFIG.target_gene
)
expr_filtered = filter_low_variance(
    expr_top, CONFIG.variance_threshold, ensure_gene=CONFIG.target_gene
)
print(f"Final shape: {expr_filtered.shape[0]} genes x {expr_filtered.shape[1]} samples")
print(f"TP53 in final set: {CONFIG.target_gene in expr_filtered.index}")

18:40:59 | INFO | Selected top 5000 variable genes
18:40:59 | INFO | Filtered genes: 2001 -> 2001


Final shape: 2001 genes x 181 samples
TP53 in final set: True


In [22]:
print("Top 20 highest variance genes:")
variance_stats.head(20)

Top 20 highest variance genes:


,Gene,Variance
Gene,,
KRT6A,KRT6A,15.590383
SPRR1B,SPRR1B,10.886695
SCGB1A1,SCGB1A1,10.199984
RPS4Y1,RPS4Y1,10.007082
SPINK1,SPINK1,9.947358
KRT5,KRT5,9.807973
AKR1B10,AKR1B10,9.746627
SCGB3A2,SCGB3A2,9.128093
SFTA2,SFTA2,9.008272


## 4. Unit Tests

In [23]:
def test_log2_transform():
    test_df = pd.DataFrame(
        {"S1": [0, 1, 3], "S2": [7, 15, 31]}, index=["G1", "G2", "G3"]
    )
    result = apply_log2_transform(test_df)
    assert np.isclose(result.loc["G1", "S1"], 0.0)
    assert np.isclose(result.loc["G1", "S2"], 3.0)


def test_tp53_included():
    assert CONFIG.target_gene in expr_filtered.index, f"{CONFIG.target_gene} missing!"
    print(f"OK: TP53 confirmed in final gene set!")


test_log2_transform()
test_tp53_included()
logging.info("All tests passed.")

18:40:59 | INFO | Applied log2(x+1) transformation
18:40:59 | INFO | All tests passed.


OK: TP53 confirmed in final gene set!


## 5. Export Results

In [24]:
preprocessed_path = CONFIG.export_dir / "task1_preprocessed_expression.csv"
expr_filtered.to_csv(preprocessed_path)
logging.info(
    "[OK] Preprocessed expression matrix saved to: %s", preprocessed_path.resolve()
)

variance_stats.to_csv(CONFIG.export_dir / "task1_variance_stats.csv", index=False)
logging.info("[OK] Variance statistics saved.")

if CONFIG.target_gene in expr_filtered.index:
    tp53_profile = expr_filtered.loc[CONFIG.target_gene]
    tp53_profile.to_csv(CONFIG.export_dir / "task1_tp53_expression.csv", header=True)
    logging.info("[OK] TP53 expression profile saved.")

print(f"\n{'=' * 60}")
print("TASK 1 COMPLETE - Data Preprocessing")
print(f"{'=' * 60}")
print(f"Dataset: GSE50081 Non-Small Cell Lung Cancer")
print(f"Genes: {expr_filtered.shape[0]} (from {expr_raw.shape[0]} original)")
print(f"Samples: {expr_filtered.shape[1]} tumor samples")
print(f"TP53 included: Yes")

18:40:59 | INFO | [OK] Preprocessed expression matrix saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/07_network_viz/assignments/artifacts/task1_preprocessed_expression.csv
18:40:59 | INFO | [OK] Variance statistics saved.
18:40:59 | INFO | [OK] TP53 expression profile saved.



TASK 1 COMPLETE - Data Preprocessing
Dataset: GSE50081 Non-Small Cell Lung Cancer
Genes: 2001 (from 2001 original)
Samples: 181 tumor samples
TP53 included: Yes
